# Deliverable 3 — RAG Pipeline (Food Delivery Knowledge Base)

Full pipeline: PDF ingestion → chunking (sentence-based, with overlap) → ChromaDB vector index → BM25 keyword index → RRF hybrid fusion → cross-encoder reranking → grounded answer generation **with citations** → retrieval evaluation metrics.

This follows the same stage structure used in the Day 3 lab, applied to our own knowledge base (`food delivery knowledge base.pdf`).

In [1]:
# Install the required libraries for the RAG pipeline
!pip -q install chromadb sentence-transformers rank-bm25 pypdf transformers accelerate torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [2]:
import re
import numpy as np
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from pypdf import PdfReader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

## Stage 1 — Load the knowledge base PDF

In [3]:
def load_pdf(pdf_path: str) -> str:
    """
    Reads every page of the knowledge base PDF and concatenates the text.
    """
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text() or ""
        text += page_text + "\n"
    print(f"   ✅ Loaded {len(reader.pages)} pages, {len(text)} characters")
    return text

## Stage 2 — Chunking (sentence-based, with overlap)

The original version split every 500 **characters**, which cuts sentences and even words in half and hurts both embedding quality and BM25 matching. Here we split by sentence and use a sliding window with overlap, the same approach as the doctor's example, so context is not lost at chunk boundaries.

In [4]:
def chunk_text(raw_text: str, doc_id: str = "food_delivery_kb",
               sentences_per_chunk: int = 5, overlap: int = 1) -> list[dict]:
    """
    Splits text into overlapping sentence-level chunks.
    sentences_per_chunk = 5  -> each chunk holds ~5 sentences
    overlap = 1              -> the last sentence of a chunk re-appears as the
                                  first sentence of the next chunk (context continuity)
    """
    # Clean stray artifacts left by PDF extraction (emoji markers, extra whitespace)
    clean = re.sub(r"\s+", " ", raw_text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", clean)
    sentences = [s for s in sentences if s.strip()]

    step = max(1, sentences_per_chunk - overlap)
    chunks = []
    for i in range(0, len(sentences), step):
        window = sentences[i: i + sentences_per_chunk]
        if not window:
            continue
        chunk_text_ = " ".join(window)
        chunks.append({
            "id": f"{doc_id}_chunk_{i:04d}",
            "text": chunk_text_,
            "doc_id": doc_id,
        })
        if i + sentences_per_chunk >= len(sentences):
            break
    return chunks

## Stage 3 — Vector index (ChromaDB + bi-encoder embeddings)

In [5]:
def build_vector_index(chunks: list[dict], collection_name: str = "food_delivery_knowledge_base"):
    """
    Embeds every chunk with all-MiniLM-L6-v2 and stores it in ChromaDB
    (HNSW index under the hood, same as Pinecone/Weaviate).
    get_or_create_collection is used instead of create_collection so the
    cell can be re-run without an "already exists" error.
    """
    print("\n📦 Building ChromaDB vector index...")
    ef = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
    client = chromadb.Client()
    collection = client.get_or_create_collection(collection_name, embedding_function=ef)
    collection.add(
        ids=[c["id"] for c in chunks],
        documents=[c["text"] for c in chunks],
        metadatas=[{"doc_id": c["doc_id"]} for c in chunks],
    )
    print(f"   ✅ {len(chunks)} chunks indexed (HNSW backend, all-MiniLM-L6-v2 embeddings)")
    return collection

## Stage 4 — BM25 keyword index

In [6]:
class BM25Index:
    """
    Classic keyword search (BM25). Catches exact terms / product names that
    semantic search can miss, e.g. "5°C", specific policy names, IDs.
    """
    def __init__(self, chunks: list[dict]):
        tokenised = [c["text"].lower().split() for c in chunks]
        self.bm25 = BM25Okapi(tokenised)
        self.chunks = chunks

    def search(self, query: str, top_k: int = 10) -> list[tuple[float, dict]]:
        scores = self.bm25.get_scores(query.lower().split())
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return [(score, self.chunks[idx]) for idx, score in ranked[:top_k]]

## Stage 5 — Hybrid search: Reciprocal Rank Fusion (RRF)

**This is the main fix vs the original notebook.** The RRF function existed before but was never actually called — the old "hybrid" step just concatenated the vector results and the BM25 results one after another (a plain union, not a fusion). That does **not** satisfy the rubric's "hybrid search ... fused e.g. with Reciprocal Rank Fusion" requirement. Below, RRF is defined **and used** inside the pipeline.

In [7]:
def reciprocal_rank_fusion(
    vector_hits: list[dict],
    bm25_hits: list[tuple[float, dict]],
    k: int = 60,
    top_k: int = 10,
) -> list[dict]:
    """
    RRF score = sum(1 / (k + rank_i)) across both ranked lists.
    k=60 is the standard constant. RRF needs no weight tuning and is the
    accepted way to merge a dense (vector) list with a sparse (BM25) list.
    """
    rrf_scores: dict[str, float] = {}
    id_to_chunk: dict[str, dict] = {}

    for rank, hit in enumerate(vector_hits):
        cid = hit["id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
        id_to_chunk[cid] = {"id": cid, "text": hit["document"]}

    for rank, (_, chunk) in enumerate(bm25_hits):
        cid = chunk["id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
        id_to_chunk[cid] = chunk

    sorted_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    return [id_to_chunk[cid] for cid in sorted_ids[:top_k]]

## Stage 6 — Cross-encoder reranking

**This stage was completely missing from the original notebook**, and it is worth real points on the rubric ("reranking (cross-encoder)"). A bi-encoder (the embedding model) scores the query and each chunk *independently*, which is fast but approximate. A cross-encoder scores the *pair* (query, chunk) together, which is slower but much more precise — so we use it as a second pass only on the small shortlist RRF already narrowed down.

In [8]:
def rerank(query: str, candidates: list[dict], model: CrossEncoder, top_k: int = 3) -> list[dict]:
    """
    Two-stage retrieval, stage 2: cross-encoder scores each (query, chunk)
    pair jointly. The CrossEncoder is loaded ONCE in main() and passed in
    here, instead of being re-loaded from disk/hub on every call.
    """
    print(f"  🎯 Cross-encoder reranking {len(candidates)} candidates...")
    pairs = [(query, c["text"]) for c in candidates]
    scores = model.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]

## Stage 7 — Grounded prompt construction (with citations)

The original prompt dumped raw chunk text into the context with no source labels, so the model's answer could not be traced back to a specific chunk. The rubric explicitly asks for *"answers grounded in retrieved context with citations."* Each chunk is now tagged `[Source N]`, and the generation step is instructed to cite the source number it used.

In [9]:
def build_rag_prompt(query: str, context_docs: list[dict]) -> str:
    context = "\n\n".join(
        f"[Source {i+1}]: {d['text']}" for i, d in enumerate(context_docs)
    )
    return (
        "You are an AI assistant for a food delivery platform.\n"
        "Answer the question using ONLY the information in the context below.\n"
        "After every factual statement, cite the source it came from like this: (Source 2).\n"
        "If the answer is not contained in the context, say you don't have enough information.\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {query}\n\n"
        "ANSWER:"
    )

## Stage 8 — Answer generation (FLAN-T5)

In [10]:
def generate_answer(prompt: str, tokenizer, model) -> str:
    """
    tokenizer and model are loaded ONCE in main() and passed in here,
    instead of calling from_pretrained() again for every query.
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    outputs = model.generate(**inputs, max_new_tokens=200)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

## Stage 9 — Retrieval evaluation (simplified RAGAS-style metrics)

Not explicitly required by the rubric text, but it is what the doctor's lab does to demonstrate the pipeline actually retrieves relevant context, and it strengthens the submission ("prove the failure paths, not just the happy path").

In [11]:
def evaluate(query: str, retrieved_docs: list[dict], embed_model: SentenceTransformer,
             relevance_threshold: float = 0.30) -> dict:
    """
    Context Precision = fraction of retrieved chunks whose cosine similarity
                         to the query is above the threshold (proxy for "no noise").
    Avg Similarity     = mean cosine similarity (proxy for context recall).
    """
    q_emb = embed_model.encode(query, normalize_embeddings=True)
    scores = [
        float(np.dot(q_emb, embed_model.encode(d["text"], normalize_embeddings=True)))
        for d in retrieved_docs
    ]
    relevant = sum(s > relevance_threshold for s in scores)
    return {
        "context_precision": round(relevant / len(scores), 3),
        "avg_similarity": round(sum(scores) / len(scores), 3),
        "chunks_in_context": len(retrieved_docs),
    }

## Main — full pipeline run

In [12]:
def main(pdf_path: str = "/content/food delivery knowledge base.pdf"):
    print("=" * 65)
    print("  Deliverable 3 — RAG Pipeline (Food Delivery Knowledge Base)")
    print("=" * 65)

    # Stage 1: Load
    raw_text = load_pdf(pdf_path)

    # Stage 2: Chunk
    chunks = chunk_text(raw_text)
    print(f"\n📄 Document → {len(chunks)} sentence-overlap chunks")

    # Stage 3 + 4: Build indexes
    collection = build_vector_index(chunks)
    bm25_index = BM25Index(chunks)
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")

    # Load the cross-encoder and the generation model ONCE, outside the query
    # loop. Loading these inside the loop (once per query) was the bug flagged
    # in review: it reloads ~250MB+ of weights from disk/hub on every single
    # query instead of reusing the already-loaded model in memory.
    print("\n🧠 Loading cross-encoder and generation model (once)...")
    cross_encoder_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
    gen_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    gen_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
    print("   ✅ Models loaded, reused for every query below")

    queries = [
        "What are the food safety requirements?",
        "What temperature should hot food be kept at during delivery?",
        "What payment methods are accepted?",
        "Can customers schedule an order for later?",
    ]

    for query in queries:
        print(f"\n{'=' * 65}")
        print(f"QUERY: {query}")
        print("=" * 65)

        # Vector (semantic) search
        vec_results = collection.query(query_texts=[query], n_results=10)
        vec_hits = [
            {"id": vid, "document": vdoc}
            for vid, vdoc in zip(vec_results["ids"][0], vec_results["documents"][0])
        ]
        print(f"\n  🔍 Vector search:   {len(vec_hits)} candidates")

        # BM25 keyword search
        bm25_hits = bm25_index.search(query, top_k=10)
        print(f"  🔑 BM25 search:     {len(bm25_hits)} candidates")

        # Hybrid RRF fusion (actually applied this time)
        hybrid = reciprocal_rank_fusion(vec_hits, bm25_hits, top_k=10)
        print(f"  ⚡ RRF fusion:      {len(hybrid)} merged candidates")

        # Cross-encoder rerank (reusing the already-loaded model)
        final_docs = rerank(query, hybrid, cross_encoder_model, top_k=3)

        # Build grounded + cited prompt
        prompt = build_rag_prompt(query, final_docs)
        print("\n  📋 Top-3 chunks after reranking:")
        for i, doc in enumerate(final_docs, 1):
            print(f"    [Source {i}] {doc['text'][:120]}...")

        # Generate grounded answer (reusing the already-loaded model)
        answer = generate_answer(prompt, gen_tokenizer, gen_model)
        print("\n  💬 Answer:")
        print(f"    {answer}")

        # Evaluate retrieval quality
        metrics = evaluate(query, final_docs, embed_model)
        print(f"\n  📊 Retrieval metrics: {metrics}")
        if metrics["context_precision"] < 0.5:
            print("  ⚠️  Low precision — consider a smaller chunk size or higher top_k before reranking")
        else:
            print("  ✅ Retrieval quality looks good")

    print("\n" + "=" * 65)
    print("Pipeline complete.")
    print("=" * 65)

## Run

In [13]:
main()

  Deliverable 3 — RAG Pipeline (Food Delivery Knowledge Base)
   ✅ Loaded 17 pages, 11618 characters

📄 Document → 34 sentence-overlap chunks

📦 Building ChromaDB vector index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

   ✅ Models loaded, reused for every query below

QUERY: What are the food safety requirements?

  🔍 Vector search:   10 candidates
  🔑 BM25 search:     10 candidates
  ⚡ RRF fusion:      10 merged candidates
  🎯 Cross-encoder reranking 10 candidates...

  📋 Top-3 chunks after reranking:
    [Source 1] Weather Conditions Drivers should reduce speed during: • Heavy rain • Dust storms • Fog If weather conditions become dan...
    [Source 2] During Delivery Drivers should: • Keep insulated bags closed. • Protect drinks from spilling. • Follow GPS navigation. •...
    [Source 3] During peak hours such as lunch and dinner, restaurants should regularly review estimated preparation times to avoid unr...

  💬 Answer:
    Cold food should remain below 5°C. Hot food should remain above 60°C. These temperatures reduce the risk of bacterial growth. Transport Time Orders exceeding safe transport times should be discarded rather than delivered.

  📊 Retrieval metrics: {'context_precision': 1.0, 'avg